# Revised PCBert-Kla released-data benchmark

This notebook evaluates PCBert-Kla and MM-Kla on a new, leakage-screened partition of the **same released PCBert-Kla rows**. Contradictory labels and exact duplicates are removed. Candidate protein matches and qualifying full-protein similarity are grouped before splitting. The original rows do not provide verified parent-protein IDs, so this remains a diagnostic internal benchmark, not proof of protein-independent or external-study generalization. Training is disabled by default.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys, tempfile

REPO_URL = 'https://github.com/kaleabnega/MM-Kla.git'
REPO_BRANCH = 'hybridkla-benchmark'
REPO_DIR = Path('/kaggle/working/MM-Kla')
PROJECT = REPO_DIR / 'experiment' / 'pcbert_kla_clean'
OUTPUT = Path('/kaggle/working/pcbert_revised_results')
OUTPUT.mkdir(parents=True, exist_ok=True)
RUN_PAIRED_MODELS = False
SEEDS = (42, 123, 777)
EPOCHS = 30
PATIENCE = 6


## Checkout and environment

The private repository is cloned with the Kaggle secret `GITHUB_TOKEN`. The checked-out commit and available GPUs are recorded with the results.


In [ ]:
if not REPO_DIR.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Kaggle secret GITHUB_TOKEN is missing')
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\\n" x-access-token ;;\n  *Password*) printf "%s\\n" "$MM_KLA_GITHUB_TOKEN" ;;\n  *) exit 1 ;;\nesac\n')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'MM_KLA_GITHUB_TOKEN': token})
        result = subprocess.run(['git', '-c', 'credential.helper=', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], env=env, capture_output=True, text=True)
        del token, env
    if result.returncode:
        raise RuntimeError('Private clone failed; check token access and branch')
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=REPO_DIR, text=True).strip()
if branch != REPO_BRANCH:
    raise RuntimeError(f'Expected {REPO_BRANCH}, found {branch}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=PROJECT, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('A Kaggle GPU runtime is required for training')
runtime = {'commit': commit, 'branch': branch, 'gpu': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())], 'torch': torch.__version__}
(OUTPUT / 'runtime.json').write_text(json.dumps(runtime, indent=2) + '\n')
print(json.dumps(runtime, indent=2))


## Rebuild and verify the lock

The source CSVs, candidate mapping, homology clusters, and search hits are pinned by SHA-256. Regeneration must reproduce the committed manifests byte-for-byte before any model is trained.


In [ ]:
committed = PROJECT / 'review_data' / 'pcbert_revised_v1'
regenerated = OUTPUT / 'regenerated_lock'
result = subprocess.run([sys.executable, 'scripts/lock_pcbert_revised.py', '--output-dir', str(regenerated)], cwd=PROJECT, text=True, capture_output=True)
if result.returncode:
    raise RuntimeError('Lock regeneration failed:\n' + result.stdout[-2000:] + result.stderr[-2000:])
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
for filename in ('full_manifest.csv', 'model_manifest.csv', 'split_audit.json'):
    if sha256(committed / filename) != sha256(regenerated / filename):
        raise ValueError('Regenerated lock differs from committed ' + filename)
lock = committed / 'split_audit.json'
audit = json.loads(lock.read_text())
check = subprocess.run([sys.executable, 'scripts/run_replication.py', '--run', 'pcbert-revised-check', '--pcbert-revised-lock', str(lock)], cwd=PROJECT, text=True, capture_output=True, check=True)
print(check.stdout)
print('Original test after exact train-overlap exclusion:', audit['original_test_if_train_overlaps_removed'])


## Matched model training

Three predeclared seeds use the same locked train, validation, and internal-test rows for both architectures. Each job has a 30-epoch maximum with validation-AUPRC early stopping and a fixed 0.5 decision threshold. The internal test is not used to select settings. On two GPUs, the models run concurrently, one per device.


In [ ]:
def start_run(architecture, seed, device):
    destination = OUTPUT / f'seed_{seed}' / architecture
    destination.mkdir(parents=True, exist_ok=True)
    args = [sys.executable, 'scripts/run_replication.py', '--run', 'pcbert-revised', '--pcbert-revised-lock', str(lock), '--architecture', architecture, '--model-name', 'Rostlab/prot_bert', '--optimizer', 'sgd', '--learning-rate', '0.003', '--weight-decay', '0', '--scheduler', 'none', '--epochs', str(EPOCHS), '--patience', str(PATIENCE), '--batch-size', '4', '--device', device, '--seed', str(seed), '--output-dir', str(destination)]
    logfile = (destination / 'train.log').open('w')
    process = subprocess.Popen(args, cwd=PROJECT, stdout=logfile, stderr=subprocess.STDOUT)
    return process, logfile, destination

if RUN_PAIRED_MODELS:
    for seed in SEEDS:
        jobs = {}
        try:
            for architecture in ('baseline', 'token_gated'):
                device = 'cuda:0' if architecture == 'baseline' or torch.cuda.device_count() == 1 else 'cuda:1'
                jobs[architecture] = start_run(architecture, seed, device)
                if torch.cuda.device_count() == 1:
                    process, logfile, destination = jobs.pop(architecture)
                    code = process.wait()
                    logfile.close()
                    if code:
                        raise RuntimeError(architecture + ' failed; see ' + str(destination / 'train.log'))
            for architecture, (process, logfile, destination) in list(jobs.items()):
                code = process.wait()
                logfile.close()
                del jobs[architecture]
                if code:
                    raise RuntimeError(architecture + ' failed; see ' + str(destination / 'train.log'))
        finally:
            for process, logfile, destination in jobs.values():
                if process.poll() is None:
                    process.terminate()
                    process.wait()
                logfile.close()
        print('Completed matched seed', seed)
else:
    print('Matched training disabled')


## Generated comparison

The table is populated only from completed runs. The internal-test sample size and class prevalence remain fixed across seeds and architectures; no paper-reported metric is inserted into the table.


In [ ]:
import pandas as pd
records = []
for seed in SEEDS:
    for architecture in ('baseline', 'token_gated'):
        directory = OUTPUT / f'seed_{seed}' / architecture / 'pcbert_revised' / 'runs'
        if not directory.exists():
            continue
        files = sorted(directory.glob('*/test_metrics.json'))
        if len(files) != 1:
            raise RuntimeError(f'Expected one result for {architecture}, seed {seed}')
        records.append({'seed': seed, 'architecture': architecture, **json.loads(files[0].read_text())})
results = pd.DataFrame.from_records(records)
if results.empty:
    print('No completed model evaluations')
else:
    display(results)
    results.to_csv(OUTPUT / 'paired_internal_test.csv', index=False)
    if set(results['seed']) == set(SEEDS) and len(results) == 2 * len(SEEDS):
        summary = results.groupby('architecture')[['AUPRC', 'AUC', 'MCC', 'F1', 'Pre', 'Rec', 'SP', 'ACC']].agg(['mean', 'std'])
        display(summary)
        summary.to_csv(OUTPUT / 'paired_summary.csv')


## Internal-test uncertainty

Whole candidate-homology groups are resampled to give descriptive 95% intervals for AUPRC and AUC. These intervals do not make the retrospective test independent.


In [ ]:
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score
manifest = pd.read_csv(committed / 'model_manifest.csv')
groups_by_id = manifest.set_index('window_id')['candidate_group'].to_dict()
intervals = []
for seed in SEEDS:
    for architecture in ('baseline', 'token_gated'):
        directory = OUTPUT / f'seed_{seed}' / architecture / 'pcbert_revised' / 'runs'
        if not directory.exists():
            continue
        files = sorted(directory.glob('*/diagnostic_test_predictions.csv'))
        if len(files) != 1:
            raise RuntimeError(f'Expected one prediction file for {architecture}, seed {seed}')
        frame = pd.read_csv(files[0])
        group_ids = frame['name'].map(groups_by_id)
        if group_ids.isna().any():
            raise ValueError('Prediction window missing from locked manifest')
        groups = group_ids.to_numpy()
        labels = frame['true_label'].to_numpy(dtype=int)
        scores = frame['score'].to_numpy(dtype=float)
        unique = np.unique(groups)
        index = {group: np.flatnonzero(groups == group) for group in unique}
        rng = np.random.default_rng(2026)
        bootstrap = []
        for _ in range(2000):
            selected_groups = rng.choice(unique, size=len(unique), replace=True)
            selected = np.concatenate([index[group] for group in selected_groups])
            if len(np.unique(labels[selected])) < 2:
                continue
            bootstrap.append((average_precision_score(labels[selected], scores[selected]), roc_auc_score(labels[selected], scores[selected])))
        if len(bootstrap) < 1900:
            raise RuntimeError('Too many degenerate group-bootstrap samples')
        bounds = np.percentile(np.asarray(bootstrap), [2.5, 97.5], axis=0)
        intervals.append({'seed': seed, 'architecture': architecture, 'positive_test_windows': int(labels.sum()), 'AUPRC_low': bounds[0, 0], 'AUPRC_high': bounds[1, 0], 'AUC_low': bounds[0, 1], 'AUC_high': bounds[1, 1]})
if intervals:
    display(pd.DataFrame(intervals))
    pd.DataFrame(intervals).to_csv(OUTPUT / 'group_bootstrap_intervals.csv', index=False)
else:
    print('No completed prediction files')


## Scope

This diagnostic answers exact-window duplication and candidate-protein similarity at the declared threshold. Candidate accessions are sequence matches, **not verified original parent IDs**; the original released benchmark cannot establish a fully protein-independent or new-study test. The held-out test here is an internal repartition of old data. These rows were already examined during earlier model development, so the repartition is retrospective, not freshly blinded. External validation requires a separate accession-labelled source.
